# MRC 데이터셋 및 Corpus 데이터 품질 점검

이 노트북은 MRC 데이터셋과 Corpus 데이터의 신뢰성과 정합성을 확보하기 위한 포괄적인 점검을 수행합니다.


## 1. 라이브러리 임포트


In [35]:
# 가상환경(.venv) 경로 자동 추가import sysfrom pathlib import Path# 프로젝트 루트 찾기project_root = Path().resolve().parent.parentvenv_path = project_root / ".venv"if venv_path.exists():    # Python 버전에 맞는 site-packages 경로 찾기    python_version = f"{sys.version_info.major}.{sys.version_info.minor}"    venv_site_packages = venv_path / "lib" / f"python{python_version}" / "site-packages"        # site-packages가 없으면 다른 가능한 경로 시도    if not venv_site_packages.exists():        lib_dir = venv_path / "lib"        if lib_dir.exists():            for py_dir in lib_dir.iterdir():                if py_dir.is_dir() and py_dir.name.startswith("python"):                    site_packages = py_dir / "site-packages"                    if site_packages.exists():                        venv_site_packages = site_packages                        break        if venv_site_packages.exists():        venv_path_str = str(venv_site_packages)        if venv_path_str not in sys.path:            sys.path.insert(0, venv_path_str)        print(f"✅ 가상환경(.venv) 경로 추가됨: {venv_site_packages}")    else:        print(f"⚠️ 가상환경(.venv)이 존재하지만 site-packages를 찾을 수 없습니다")else:    print(f"ℹ️ 가상환경(.venv)이 없습니다. 시스템 Python을 사용합니다")
# 노트북 독립 실행을 위한 환경 설정
import sys
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 공통 유틸리티 import
try:
    from notebooks.utils import setup_notebook_environment, load_dataset_safely, load_json_safely
    
    # 환경 설정
    paths = setup_notebook_environment()
    print(f"✅ 프로젝트 루트: {paths['project_root']}")
    print(f"✅ 데이터 디렉토리: {paths['data_dir']}")
except ImportError as e:
    print(f"⚠️ 유틸리티 import 실패: {e}")
    paths = {
        'project_root': project_root,
        'data_dir': project_root / 'data',
        'notebook_dir': project_root / 'notebooks'
    }


from datasets import load_from_disk
import pandas as pd
import numpy as np
import json
import re
from collections import Counter
from tqdm import tqdm

print("라이브러리 임포트 완료")


라이브러리 임포트 완료


## 2. 데이터 로드


In [36]:
# MRC 데이터셋 로드
print("MRC 데이터셋 로드 중...")
train_dataset = load_from_disk("../../data/train_dataset")
train_df = pd.DataFrame(train_dataset["train"])
val_df = pd.DataFrame(train_dataset["validation"])

print(f"Train 데이터: {len(train_df)}개")
print(f"Validation 데이터: {len(val_df)}개")
print(f"\n컬럼: {train_df.columns.tolist()}")

# Corpus 데이터 로드
print("\nCorpus 데이터 로드 중...")
with open("../../data/wikipedia_documents.json", "r", encoding="utf-8") as f:
    corpus_data = json.load(f)

corpus_list = [v for k, v in sorted(corpus_data.items(), key=lambda x: int(x[0]))]
corpus_df = pd.DataFrame(corpus_list)

print(f"Corpus 문서 개수: {len(corpus_df)}개")
print(f"\n컬럼: {corpus_df.columns.tolist()}")


MRC 데이터셋 로드 중...
Train 데이터: 3952개
Validation 데이터: 240개

컬럼: ['title', 'context', 'question', 'id', 'answers', 'document_id', '__index_level_0__']

Corpus 데이터 로드 중...
Corpus 문서 개수: 60613개

컬럼: ['text', 'corpus_source', 'url', 'domain', 'title', 'author', 'html', 'document_id']


# 3. MRC 데이터셋 점검

## 3.1 기본 무결성 검사


In [37]:
# 3.1.1 id 값 중복 확인
train_duplicate_ids = train_df[train_df.duplicated(subset=['id'], keep=False)]
val_duplicate_ids = val_df[val_df.duplicated(subset=['id'], keep=False)]

print("=" * 60)
print("3.1.1 ID 중복 확인")
print("=" * 60)
print(f"Train 데이터 중복 ID 개수: {len(train_duplicate_ids)}")
print(f"Validation 데이터 중복 ID 개수: {len(val_duplicate_ids)}")

if len(train_duplicate_ids) > 0:
    print(f"\nTrain 중복 ID 목록:")
    print(train_duplicate_ids[['id', 'question']].head(10))
    
if len(val_duplicate_ids) > 0:
    print(f"\nValidation 중복 ID 목록:")
    print(val_duplicate_ids[['id', 'question']].head(10))


3.1.1 ID 중복 확인
Train 데이터 중복 ID 개수: 0
Validation 데이터 중복 ID 개수: 0


In [38]:
# 3.1.2 필수 필드 검증
required_fields = ['id', 'question', 'context', 'answers']

def check_required_fields(df, dataset_name):
    missing_fields = []
    for field in required_fields:
        if field not in df.columns:
            missing_fields.append(field)
        else:
            # None 또는 NaN 값 확인
            null_count = df[field].isna().sum()
            if null_count > 0:
                missing_fields.append(f"{field} (None/NaN: {null_count}개)")
    
    return missing_fields

train_missing = check_required_fields(train_df, "Train")
val_missing = check_required_fields(val_df, "Validation")

print("=" * 60)
print("3.1.2 필수 필드 검증")
print("=" * 60)
print(f"Train 데이터 누락 필드: {train_missing if train_missing else '없음'}")
print(f"Validation 데이터 누락 필드: {val_missing if val_missing else '없음'}")


3.1.2 필수 필드 검증
Train 데이터 누락 필드: 없음
Validation 데이터 누락 필드: 없음


## 3.2 Answer 검증


In [39]:
# 3.2.1 answer_start가 context의 유효한 범위에 위치하는지 확인
def check_answer_start_range(df, dataset_name):
    invalid_start_indices = []
    invalid_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        context_len = len(context)
        
        if isinstance(answers, dict) and 'answer_start' in answers:
            starts = answers['answer_start']
            if isinstance(starts, list):
                for start in starts:
                    if start < 0 or start >= context_len:
                        invalid_start_indices.append(idx)
                        invalid_details.append({
                            'id': row['id'],
                            'answer_start': start,
                            'context_len': context_len,
                            'is_negative': start < 0,
                            'is_out_of_range': start >= context_len
                        })
                        break
    
    return invalid_start_indices, invalid_details

train_invalid_start, train_start_details = check_answer_start_range(train_df, "Train")
val_invalid_start, val_start_details = check_answer_start_range(val_df, "Validation")

print("=" * 60)
print("3.2.1 answer_start 범위 확인")
print("=" * 60)
print(f"Train 데이터 유효하지 않은 answer_start: {len(train_invalid_start)}개")
print(f"Validation 데이터 유효하지 않은 answer_start: {len(val_invalid_start)}개")

if len(train_start_details) > 0:
    print(f"\nTrain 문제 샘플 (최대 5개):")
    for detail in train_start_details[:5]:
        print(f"  ID: {detail['id']}, answer_start: {detail['answer_start']}, context_len: {detail['context_len']}")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 13414.44it/s]

3.2.1 answer_start 범위 확인
Train 데이터 유효하지 않은 answer_start: 0개
Validation 데이터 유효하지 않은 answer_start: 0개


In [40]:
# 3.2.2 context[answer_start:answer_start+len(answer_text)]와 실제 answer text 일치 여부 확인
def check_answer_text_match(df, dataset_name):
    mismatch_indices = []
    mismatch_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0 or start >= len(context):
                        continue
                    
                    # answer_start 위치에서 추출한 텍스트
                    extracted_text = context[start:start+len(text)]
                    
                    # 정확히 일치하는지 확인
                    if extracted_text != text:
                        mismatch_indices.append(idx)
                        mismatch_details.append({
                            'id': row['id'],
                            'expected': text,
                            'extracted': extracted_text,
                            'start': start,
                            'context_snippet': context[max(0, start-30):start+len(text)+30]
                        })
                        break
    
    return mismatch_indices, mismatch_details

train_mismatch, train_mismatch_details = check_answer_text_match(train_df, "Train")
val_mismatch, val_mismatch_details = check_answer_text_match(val_df, "Validation")

print("=" * 60)
print("3.2.2 Answer Text 일치 여부 확인")
print("=" * 60)
print(f"Train 데이터 불일치: {len(train_mismatch)}개")
print(f"Validation 데이터 불일치: {len(val_mismatch)}개")

if len(train_mismatch_details) > 0:
    print(f"\nTrain 불일치 샘플 (최대 5개):")
    for i, detail in enumerate(train_mismatch_details[:5]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer start: {detail['start']}")
        print(f"    예상 텍스트: '{detail['expected']}'")
        print(f"    추출된 텍스트: '{detail['extracted']}'")
        print(f"    Context 주변: ...{detail['context_snippet']}...")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 20603.24it/s]

3.2.2 Answer Text 일치 여부 확인
Train 데이터 불일치: 0개
Validation 데이터 불일치: 0개


In [41]:
# 3.2.3 Answer text가 빈 문자열이 아닌지 점검
def check_empty_answers(df, dataset_name):
    empty_answer_indices = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        if isinstance(answers, dict) and 'text' in answers:
            texts = answers['text']
            if isinstance(texts, list):
                # 빈 문자열이 있는지 확인
                if any(text == '' or text is None for text in texts):
                    empty_answer_indices.append(idx)
    
    return empty_answer_indices

train_empty_answers = check_empty_answers(train_df, "Train")
val_empty_answers = check_empty_answers(val_df, "Validation")

print("=" * 60)
print("3.2.3 빈 Answer Text 확인")
print("=" * 60)
print(f"Train 데이터 빈 answer: {len(train_empty_answers)}개")
print(f"Validation 데이터 빈 answer: {len(val_empty_answers)}개")

if len(train_empty_answers) > 0:
    print(f"\nTrain 빈 answer 샘플:")
    print(train_df.loc[train_empty_answers, ['id', 'question', 'answers']].head(10))


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 13736.99it/s]

3.2.3 빈 Answer Text 확인
Train 데이터 빈 answer: 0개
Validation 데이터 빈 answer: 0개


In [42]:
# 3.2.4 Answer 양 끝의 불필요한 공백으로 인한 mismatch 여부 확인
def check_answer_whitespace_mismatch(df, dataset_name):
    whitespace_mismatch_indices = []
    whitespace_mismatch_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0 or start >= len(context):
                        continue
                    
                    # 원본 추출
                    extracted = context[start:start+len(text)]
                    
                    # 정규화된 버전 비교 (공백 정규화)
                    normalized_extracted = re.sub(r'\s+', ' ', extracted.strip())
                    normalized_text = re.sub(r'\s+', ' ', text.strip())
                    
                    # 원본은 다르지만 정규화 후 같으면 whitespace 문제
                    if extracted != text and normalized_extracted == normalized_text:
                        whitespace_mismatch_indices.append(idx)
                        whitespace_mismatch_details.append({
                            'id': row['id'],
                            'answer_text': text,
                            'extracted': extracted,
                            'answer_start': start,
                            'has_leading_space': text != text.lstrip() or extracted != extracted.lstrip(),
                            'has_trailing_space': text != text.rstrip() or extracted != extracted.rstrip()
                        })
                        break
    
    return whitespace_mismatch_indices, whitespace_mismatch_details

train_ws_mismatch, train_ws_details = check_answer_whitespace_mismatch(train_df, "Train")
val_ws_mismatch, val_ws_details = check_answer_whitespace_mismatch(val_df, "Validation")

print("=" * 60)
print("3.2.4 Answer 공백 Mismatch 확인")
print("=" * 60)
print(f"Train 데이터 공백 mismatch: {len(train_ws_mismatch)}개")
print(f"Validation 데이터 공백 mismatch: {len(val_ws_mismatch)}개")

if len(train_ws_details) > 0:
    print(f"\nTrain 공백 mismatch 샘플 (최대 5개):")
    for i, detail in enumerate(train_ws_details[:5]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer text: {repr(detail['answer_text'])}")
        print(f"    Extracted: {repr(detail['extracted'])}")
        print(f"    앞 공백: {detail['has_leading_space']}, 뒤 공백: {detail['has_trailing_space']}")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 17804.85it/s]

3.2.4 Answer 공백 Mismatch 확인
Train 데이터 공백 mismatch: 0개
Validation 데이터 공백 mismatch: 0개


## 3.3 Context 검증


In [43]:
# 3.3.1 context가 비어 있지 않은지 확인
train_df['context_length'] = train_df['context'].apply(len)
val_df['context_length'] = val_df['context'].apply(len)

train_empty_context = train_df[train_df['context_length'] == 0]
val_empty_context = val_df[val_df['context_length'] == 0]

print("=" * 60)
print("3.3.1 빈 Context 확인")
print("=" * 60)
print(f"Train 데이터 빈 context: {len(train_empty_context)}개")
print(f"Validation 데이터 빈 context: {len(val_empty_context)}개")
print(f"\nTrain 평균 context 길이: {train_df['context_length'].mean():.2f}자")
print(f"Validation 평균 context 길이: {val_df['context_length'].mean():.2f}자")

if len(train_empty_context) > 0:
    print(f"\nTrain 빈 context 샘플:")
    print(train_empty_context[['id', 'question']].head(10))


3.3.1 빈 Context 확인
Train 데이터 빈 context: 0개
Validation 데이터 빈 context: 0개

Train 평균 context 길이: 920.22자
Validation 평균 context 길이: 916.73자


In [44]:
# 3.3.2 HTML 태그나 불필요한 마크업이 포함되어 있는지 점검
def check_html_markup(df, dataset_name):
    html_pattern = re.compile(r'<[^>]+>')
    markup_pattern = re.compile(r'&[a-z]+;|&#\d+;')
    
    markup_indices = []
    markup_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        context = str(row['context'])
        
        html_tags = html_pattern.findall(context)
        markup_entities = markup_pattern.findall(context)
        
        if html_tags or markup_entities:
            markup_indices.append(idx)
            markup_details.append({
                'id': row['id'],
                'html_tags': len(html_tags),
                'markup_entities': len(markup_entities),
                'sample_tags': html_tags[:3] if html_tags else [],
                'sample_entities': markup_entities[:3] if markup_entities else []
            })
    
    return markup_indices, markup_details

train_markup, train_markup_details = check_html_markup(train_df, "Train")
val_markup, val_markup_details = check_html_markup(val_df, "Validation")

print("=" * 60)
print("3.3.2 HTML 태그/Markup 포함 여부 확인")
print("=" * 60)
print(f"Train 데이터 markup 포함: {len(train_markup)}개")
print(f"Validation 데이터 markup 포함: {len(val_markup)}개")

if len(train_markup_details) > 0:
    print(f"\nTrain markup 포함 샘플 (최대 5개):")
    for i, detail in enumerate(train_markup_details[:5]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    HTML 태그 개수: {detail['html_tags']}")
        print(f"    Markup 엔티티 개수: {detail['markup_entities']}")
        if detail['sample_tags']:
            print(f"    샘플 태그: {detail['sample_tags']}")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 14886.40it/s]

3.3.2 HTML 태그/Markup 포함 여부 확인
Train 데이터 markup 포함: 91개
Validation 데이터 markup 포함: 3개

Train markup 포함 샘플 (최대 5개):

[1] ID: mrc-0-005458
    HTML 태그 개수: 4
    Markup 엔티티 개수: 0
    샘플 태그: ['<미들랜즈 투데이>', '<도그 쇼>', '<데니스 매카시의 위클리 에코>']

[2] ID: mrc-0-004214
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<채색화 기법>']

[3] ID: mrc-0-002418
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<알마게스트의 발췌본>']

[4] ID: mrc-0-001141
    HTML 태그 개수: 2
    Markup 엔티티 개수: 0
    샘플 태그: ['<낚시>', '<담배피우는 남자(폭포)>']

[5] ID: mrc-0-005356
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<채색화 기법>']


## 3.4 데이터 일관성 검사


In [45]:
# 3.4.1 동일 (question, context) 조합이 중복으로 존재하는지 확인
def check_duplicate_question_context_pairs(df, dataset_name):
    df_with_pair = df.copy()
    df_with_pair['question_context_pair'] = df_with_pair['question'] + '|||' + df_with_pair['context']
    
    duplicates = df_with_pair[df_with_pair.duplicated(subset=['question_context_pair'], keep=False)]
    
    return duplicates

train_dup_pairs = check_duplicate_question_context_pairs(train_df, "Train")
val_dup_pairs = check_duplicate_question_context_pairs(val_df, "Validation")

print("=" * 60)
print("3.4.1 중복된 (Question, Context) 쌍 확인")
print("=" * 60)
print(f"Train 데이터 중복 쌍: {len(train_dup_pairs)}개")
print(f"Validation 데이터 중복 쌍: {len(val_dup_pairs)}개")

if len(train_dup_pairs) > 0:
    print(f"\nTrain 중복 쌍 샘플 (최대 10개):")
    print(train_dup_pairs[['id', 'question', 'context_length']].head(10))
    
    # 중복 그룹별 개수
    pair_counts = train_dup_pairs.groupby('question_context_pair').size()
    print(f"\n중복 그룹별 개수 (상위 5개):")
    print(pair_counts.sort_values(ascending=False).head(5))


3.4.1 중복된 (Question, Context) 쌍 확인
Train 데이터 중복 쌍: 0개
Validation 데이터 중복 쌍: 0개


In [46]:
# 3.4.2 Answer text가 실제로 context 내에서 최소 1회 이상 등장하는지 검증
def check_answer_occurrence_in_context(df, dataset_name):
    not_found_indices = []
    not_found_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"{dataset_name} 체크"):
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'text' in answers:
            texts = answers['text']
            
            if isinstance(texts, list):
                for text in texts:
                    if text and text not in context:
                        not_found_indices.append(idx)
                        not_found_details.append({
                            'id': row['id'],
                            'answer_text': text,
                            'context_preview': context[:200] if len(context) > 200 else context
                        })
                        break
    
    return not_found_indices, not_found_details

train_not_found, train_not_found_details = check_answer_occurrence_in_context(train_df, "Train")
val_not_found, val_not_found_details = check_answer_occurrence_in_context(val_df, "Validation")

print("=" * 60)
print("3.4.2 Answer가 Context에 존재하는지 확인")
print("=" * 60)
print(f"Train 데이터 context에 없는 answer: {len(train_not_found)}개")
print(f"Validation 데이터 context에 없는 answer: {len(val_not_found)}개")

if len(train_not_found_details) > 0:
    print(f"\nTrain 문제 샘플 (최대 5개):")
    for i, detail in enumerate(train_not_found_details[:5]):
        print(f"\n[{i+1}] ID: {detail['id']}")
        print(f"    Answer text: '{detail['answer_text']}'")
        print(f"    Context 미리보기: {detail['context_preview']}...")


Validation 체크: 100%|██████████| 240/240 [00:00<00:00, 17975.91it/s]

3.4.2 Answer가 Context에 존재하는지 확인
Train 데이터 context에 없는 answer: 0개
Validation 데이터 context에 없는 answer: 0개


# 4. Corpus 데이터 점검

## 4.1 기본 통계


In [47]:
# 4.1.1 전체 문서 개수 파악
print("=" * 60)
print("4.1.1 전체 문서 개수")
print("=" * 60)
print(f"전체 문서 개수: {len(corpus_df)}개")


4.1.1 전체 문서 개수
전체 문서 개수: 60613개


In [48]:
# 4.1.2 문서 길이에 대한 기본 통계 확인
corpus_df['text_length'] = corpus_df['text'].apply(len)

print("=" * 60)
print("4.1.2 문서 길이 통계")
print("=" * 60)
print(f"평균 문서 길이: {corpus_df['text_length'].mean():.2f}자")
print(f"중앙값 문서 길이: {corpus_df['text_length'].median():.0f}자")
print(f"최소 문서 길이: {corpus_df['text_length'].min()}자")
print(f"최대 문서 길이: {corpus_df['text_length'].max()}자")
print(f"표준편차: {corpus_df['text_length'].std():.2f}자")

# 분위수 정보
print(f"\n분위수 정보:")
print(f"  25%: {corpus_df['text_length'].quantile(0.25):.0f}자")
print(f"  50%: {corpus_df['text_length'].quantile(0.50):.0f}자")
print(f"  75%: {corpus_df['text_length'].quantile(0.75):.0f}자")
print(f"  95%: {corpus_df['text_length'].quantile(0.95):.0f}자")


4.1.2 문서 길이 통계
평균 문서 길이: 755.57자
중앙값 문서 길이: 577자
최소 문서 길이: 184자
최대 문서 길이: 46099자
표준편차: 762.96자

분위수 정보:
  25%: 414자
  50%: 577자
  75%: 857자
  95%: 1734자


## 4.2 데이터 무결성 검사


In [49]:
# 4.2.1 내용이 비어 있는 문서 존재 여부 확인
empty_documents = corpus_df[corpus_df['text_length'] == 0]

print("=" * 60)
print("4.2.1 빈 문서 확인")
print("=" * 60)
print(f"빈 문서 개수: {len(empty_documents)}개")

if len(empty_documents) > 0:
    print(f"\n빈 문서 샘플:")
    print(empty_documents[['document_id', 'title']].head(10))


4.2.1 빈 문서 확인
빈 문서 개수: 0개


In [50]:
# 4.2.2 document_id의 중복 여부 점검
duplicate_document_ids = corpus_df[corpus_df.duplicated(subset=['document_id'], keep=False)]

print("=" * 60)
print("4.2.2 document_id 중복 확인")
print("=" * 60)
print(f"중복 document_id 개수: {len(duplicate_document_ids)}개")

if len(duplicate_document_ids) > 0:
    print(f"\n중복 document_id 목록:")
    print(duplicate_document_ids[['document_id', 'title']].head(10))
    
    # 중복 ID별 개수
    duplicate_id_counts = corpus_df['document_id'].value_counts()
    duplicate_id_counts = duplicate_id_counts[duplicate_id_counts > 1]
    print(f"\n중복된 document_id별 개수:")
    print(duplicate_id_counts.head(10))


4.2.2 document_id 중복 확인
중복 document_id 개수: 0개


## 4.3 데이터 품질 검사


In [51]:
# 4.3.1 HTML 태그나 불필요한 Markup이 포함되어 있는지 확인
def check_corpus_html_markup(df):
    html_pattern = re.compile(r'<[^>]+>')
    markup_pattern = re.compile(r'&[a-z]+;|&#\d+;')
    
    markup_indices = []
    markup_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Corpus 체크"):
        text = str(row['text'])
        
        html_tags = html_pattern.findall(text)
        markup_entities = markup_pattern.findall(text)
        
        if html_tags or markup_entities:
            markup_indices.append(idx)
            markup_details.append({
                'document_id': row['document_id'],
                'title': row.get('title', 'N/A'),
                'html_tags': len(html_tags),
                'markup_entities': len(markup_entities),
                'sample_tags': html_tags[:3] if html_tags else [],
                'sample_entities': markup_entities[:3] if markup_entities else []
            })
    
    return markup_indices, markup_details

corpus_markup, corpus_markup_details = check_corpus_html_markup(corpus_df)

print("=" * 60)
print("4.3.1 HTML 태그/Markup 포함 여부 확인")
print("=" * 60)
print(f"Markup 포함 문서: {len(corpus_markup)}개 ({len(corpus_markup)/len(corpus_df)*100:.2f}%)")

if len(corpus_markup_details) > 0:
    print(f"\nMarkup 포함 문서 샘플 (최대 10개):")
    for i, detail in enumerate(corpus_markup_details[:10]):
        print(f"\n[{i+1}] document_id: {detail['document_id']}, title: {detail['title']}")
        print(f"    HTML 태그 개수: {detail['html_tags']}")
        print(f"    Markup 엔티티 개수: {detail['markup_entities']}")
        if detail['sample_tags']:
            print(f"    샘플 태그: {detail['sample_tags']}")


Corpus 체크: 100%|██████████| 60613/60613 [00:04<00:00, 14441.41it/s]

4.3.1 HTML 태그/Markup 포함 여부 확인
Markup 포함 문서: 2186개 (3.61%)

Markup 포함 문서 샘플 (최대 10개):

[1] document_id: 37, title: 가타카나
    HTML 태그 개수: 3
    Markup 엔티티 개수: 0
    샘플 태그: ['<우쓰호모노가타리>', '<츠츠미추나곤모노가타리>', '<일영어림집성>']

[2] document_id: 50, title: 아널드 슈워제네거
    HTML 태그 개수: 5
    Markup 엔티티 개수: 0
    샘플 태그: ['<코난>', '<터미네이터>', '<토탈리콜>']

[3] document_id: 51, title: 아널드 슈워제네거
    HTML 태그 개수: 14
    Markup 엔티티 개수: 0
    샘플 태그: ['<뉴욕의 헤라클레스>', '<터미네이터>', '<터미네이터 2: 심판의 날>']

[4] document_id: 60, title: PC통신
    HTML 태그 개수: 6
    Markup 엔티티 개수: 0
    샘플 태그: ['<<<알립니다>', '<<KETEL 정보서비스 일시 정지 안내>', '<\n\u30004.>']

[5] document_id: 90, title: 웹사이트
    HTML 태그 개수: 10
    Markup 엔티티 개수: 0
    샘플 태그: ['<frameset>', '<frame>', '<frameset cols="50%,50%">']

[6] document_id: 195, title: 마오둔
    HTML 태그 개수: 3
    Markup 엔티티 개수: 0
    샘플 태그: ['<공산당>', '<소설월보>', '<신청년>']

[7] document_id: 280, title: 레프 톨스토이
    HTML 태그 개수: 1
    Markup 엔티티 개수: 0
    샘플 태그: ['<하나님 나라는 당신 안에 있다>']

[8] document_id: 281, titl

In [52]:
# 4.3.2 텍스트가 인코딩 오류 등으로 인해 비정상적으로 손상된 경우 존재 여부 점검
def check_encoding_errors(df):
    encoding_error_indices = []
    encoding_error_details = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="인코딩 오류 체크"):
        text = str(row['text'])
        has_error = False
        error_type = None
        error_count = 0
        
        # Unicode 대체 문자 확인
        if '\ufffd' in text:
            has_error = True
            error_type = 'replacement_char'
            error_count = text.count('\ufffd')
        
        # 이스케이프된 바이트 확인
        elif '\\x' in text:
            escape_pattern = re.compile(r'\\x[0-9a-f]{2}')
            matches = escape_pattern.findall(text)
            if matches:
                has_error = True
                error_type = 'escape_sequence'
                error_count = len(matches)
        
        if has_error:
            encoding_error_indices.append(idx)
            encoding_error_details.append({
                'document_id': row['document_id'],
                'title': row.get('title', 'N/A'),
                'error_type': error_type,
                'error_count': error_count,
                'sample_text': text[:200] if len(text) > 200 else text
            })
    
    return encoding_error_indices, encoding_error_details

corpus_encoding_errors, corpus_encoding_details = check_encoding_errors(corpus_df)

print("=" * 60)
print("4.3.2 인코딩 오류 확인")
print("=" * 60)
print(f"인코딩 오류 문서: {len(corpus_encoding_errors)}개 ({len(corpus_encoding_errors)/len(corpus_df)*100:.2f}%)")

if len(corpus_encoding_details) > 0:
    print(f"\n인코딩 오류 문서 샘플 (최대 10개):")
    for i, detail in enumerate(corpus_encoding_details[:10]):
        print(f"\n[{i+1}] document_id: {detail['document_id']}, title: {detail['title']}")
        print(f"    오류 유형: {detail['error_type']}")
        print(f"    오류 개수: {detail['error_count']}")
        print(f"    샘플 텍스트: {detail['sample_text'][:150]}...")


인코딩 오류 체크: 100%|██████████| 60613/60613 [00:04<00:00, 15067.87it/s]

4.3.2 인코딩 오류 확인
인코딩 오류 문서: 2개 (0.00%)

인코딩 오류 문서 샘플 (최대 10개):

[1] document_id: 29488, title: 이스케이프 시퀀스
    오류 유형: escape_sequence
    오류 개수: 3
    샘플 텍스트: 펄 또는 파이썬 2의 경우
: 
print "Nancy said "Hello World!" to the crowd.";

문법 오류를 발생시키는 반면 다음은:
: 
print "Nancy said \"Hello World!\" to the crowd.";  ### ex...

[2] document_id: 44488, title: 울산 등용사 고봉화상선요
    오류 유형: replacement_char
    오류 개수: 1
    샘플 텍스트: 고봉화상선요는 중국 송나라원대의 선승인 고봉 원묘(高峰原妙, 1238~1295)의 고봉대사어록(高峰大師語錄) 상�하권 중에서 법어와 서간을 수록한 상권의 내용을 지정(持正)이 집록(集錄)하고 직옹거사(直翁居士) 홍교조(洪喬祖)가 편집한 책이다. 내용은 수행에 정진하여 ...


## 5. 종합 요약


In [53]:
# MRC 데이터셋 점검 요약
print("=" * 60)
print("MRC 데이터셋 점검 요약")
print("=" * 60)

mrc_summary = {
    'Train': {
        '총 개수': len(train_df),
        '중복 ID': len(train_duplicate_ids),
        '누락 필드': len(train_missing),
        '유효하지 않은 answer_start': len(train_invalid_start),
        'Answer text 불일치': len(train_mismatch),
        '빈 answer': len(train_empty_answers),
        '공백 mismatch': len(train_ws_mismatch),
        '빈 context': len(train_empty_context),
        'HTML/Markup 포함': len(train_markup),
        '중복 (Q,C) 쌍': len(train_dup_pairs),
        'Context에 없는 answer': len(train_not_found)
    },
    'Validation': {
        '총 개수': len(val_df),
        '중복 ID': len(val_duplicate_ids),
        '누락 필드': len(val_missing),
        '유효하지 않은 answer_start': len(val_invalid_start),
        'Answer text 불일치': len(val_mismatch),
        '빈 answer': len(val_empty_answers),
        '공백 mismatch': len(val_ws_mismatch),
        '빈 context': len(val_empty_context),
        'HTML/Markup 포함': len(val_markup),
        '중복 (Q,C) 쌍': len(val_dup_pairs),
        'Context에 없는 answer': len(val_not_found)
    }
}

mrc_summary_df = pd.DataFrame(mrc_summary)
print(mrc_summary_df)


MRC 데이터셋 점검 요약
                      Train  Validation
총 개수                   3952         240
중복 ID                     0           0
누락 필드                     0           0
유효하지 않은 answer_start      0           0
Answer text 불일치           0           0
빈 answer                  0           0
공백 mismatch               0           0
빈 context                 0           0
HTML/Markup 포함           91           3
중복 (Q,C) 쌍                0           0
Context에 없는 answer        0           0


In [54]:
# Corpus 데이터 점검 요약
print("\n" + "=" * 60)
print("Corpus 데이터 점검 요약")
print("=" * 60)

corpus_summary = {
    '값': [
        len(corpus_df),
        f"{corpus_df['text_length'].mean():.2f}자",
        f"{corpus_df['text_length'].min()}자",
        f"{corpus_df['text_length'].max()}자",
        len(empty_documents),
        len(duplicate_document_ids),
        len(corpus_markup),
        len(corpus_encoding_errors)
    ]
}

corpus_summary_df = pd.DataFrame({
    '항목': [
        '전체 문서 개수',
        '평균 문서 길이',
        '최소 문서 길이',
        '최대 문서 길이',
        '빈 문서 개수',
        '중복 document_id 개수',
        'HTML/Markup 포함 문서',
        '인코딩 오류 문서'
    ],
    '값': corpus_summary['값']
})

print(corpus_summary_df.to_string(index=False))



Corpus 데이터 점검 요약
               항목       값
         전체 문서 개수   60613
         평균 문서 길이 755.57자
         최소 문서 길이    184자
         최대 문서 길이  46099자
          빈 문서 개수       0
중복 document_id 개수       0
HTML/Markup 포함 문서    2186
        인코딩 오류 문서       2
